In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import seaborn as sns
from sklearn.model_selection import KFold
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)



In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(df['Delivery_Time'], bins=60, edgecolor='black')
plt.title('Distribution of delivery_time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Write your code here:
# Check missing values
missing_values = df.isnull().sum()
print("Columns with missing values:")
print(missing_values[missing_values > 0])
# Fill missing values with mean for each column

# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols = ['Weather', 'Distance_km', 'Traffic_Level', 'Vehicle_Type', 'Preparation_Time_min', 'Delivery_Time']
df_clean = df[cols].copy()
# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time', 'Distance_km', 'Traffic_Level', 'Weather'])
print(f"After dropping missing price/year/odometer: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df_clean):
    duplicates = df_clean.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df_clean.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")
check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Vehicle_Type']
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col].astype(str))
df_clean.head()

In [ ]:
# Task 5: Write your code here:
feature_cols = ['Weather', 'Distance_km', 'Traffic_Level', 'Vehicle_Type', 'Preparation_Time_min']

print('data before scaling:\n', df[feature_cols]) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_standard_scaled = standard_scaler.fit_transform(df_clean[feature_cols]) # Apply fit_transform
print('\nData after scaling:\n', data_standard_scaled) #show after scaling

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()
check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X, y = df_clean[feature_cols].copy(), df_clean['Delivery_Time'].copy()

In [ ]:
# Task 2,3,4,5: Write your code here:

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
  # indexing for each fold
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
  # print shapes
  print(f"Fold {fold}")
  print("  X_train shape:", X_train.shape)
  print("  X_test shape :", X_test.shape)
  print("  y_train shape:", y_train.shape)
  print("  y_test shape :", y_test.shape)
  print("-" * 30)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
mae = mean_absolute_error(y_test, y_pred)
print("MAE :", mae)

In [ ]:
# Task 1: Write your code here:
# Plot for Linear Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Linear Regression: Predictions vs. Ground Truth")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: